In [1]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# File Paths
FILE_PATH1 = '/content/drive/MyDrive/AI351ProjectTest/04-DataAugmentation/Filtered_Dataset.csv'
FILE_PATH2 = '/content/drive/MyDrive/AI351ProjectTest/04-DataAugmentation/'

Mounted at /content/drive


In [2]:
def balanced_sample(df, sample_size, label_col="label_desc"):
    """
    Returns a balanced sample of size `sample_size`
    based on label_desc distribution.
    """
    labels = df[label_col].unique()
    n_labels = len(labels)

    # samples per class
    per_class = sample_size // n_labels
    remainder = sample_size % n_labels

    sampled_df = []

    for i, label in enumerate(labels):
        df_label = df[df[label_col] == label]
        take = per_class + (1 if i < remainder else 0)

        # If class has fewer rows than needed → sample with replacement
        sampled_df.append(df_label.sample(n=take, replace=(len(df_label) < take), random_state=42))

    return pd.concat(sampled_df).sample(frac=1, random_state=42).reset_index(drop=True)

In [3]:
main_df = pd.read_csv(FILE_PATH1)

augmentation_files = [f"Augmentation0{i}.csv" for i in range(1, 7)]
augmentation_dfs = [pd.read_csv(os.path.join(FILE_PATH2, f)) for f in augmentation_files]

# Keep only required columns
columns_to_keep = ["text", "translation", "label_desc"]

main_df = main_df[columns_to_keep]
augmentation_dfs = [df[columns_to_keep] for df in augmentation_dfs]

m = {
    0: 70,   # main dataset sample size
    1: 28,   # Augmentation01
    2: 28,   # Augmentation02
    3: 28,   # Augmentation03
    4: 28,   # Augmentation04
    5: 28,   # Augmentation05
    6: 28,   # Augmentation06
}

In [4]:
sampled_data = {}
sampled_data[0] = balanced_sample(main_df, m[0])

for i in range(1, 7):
    sampled_data[i] = balanced_sample(augmentation_dfs[i-1], m[i])

output_path = "/content/drive/MyDrive/AI351ProjectTest/06-Survey/HE.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for i in range(7):
        sheet_name = str(i)
        sampled_data[i].to_excel(writer, index=False, sheet_name=sheet_name)

print(f"Excel file successfully saved to: {output_path}")

Excel file successfully saved to: /content/drive/MyDrive/AI351ProjectTest/06-Survey/HE.xlsx
